## Getting started

In [1]:
from pathlib import Path
import spyne

Localize the data, change this as needed.

In [2]:
data_folder = Path("data")
date = "251203"
cell_n = "cell0001"

imaging_folder = data_folder / date / cell_n

## Dataset inizialization

### Imaging dataset

Let's start with the imaging dataset. This will load metadata and pre-process the imaging frame sequences.

In [3]:
dataset = spyne.ImagingDataset(imaging_folder)

✓ Loaded metadata from data\251203\cell0001\processed\imaging\metadata.h5.
✓ Loaded 170 processed arrays from: data\251203\cell0001\processed\imaging\processed_imaging.h5


If they are not already saved, let's save the loaded data and metadata (```.h5``` format) for faster fututre loading.

In [ ]:
dataset.save_metadata()
dataset.save_processed_data()

Have a glance at recorded imaging data. This will open an external window (press **q** to close it).

In [ ]:
# Indexing corresponds to [roi_n][sweep_n] (two channels by default)
dataset[6][2].show()

In [ ]:
# To see a single channel [roi number][sweep number][channel number]
dataset[6][2][0].show()

---

### Spines dataset

Now let's generate a spine dataset. This will need additional analysis (see below).

In [4]:
# Create a SpineDataset for spine analysis
spine_dataset = spyne.SpineDataset(dataset)

Loading .h5 data: 100%|██████████| 5/5 [00:00<00:00,  5.48it/s]


Let's launch spine segmentation to collect all spines masks from ROIs. By setting ```save=True``` we create predictions and metadata ```.h5``` files. In addition, these data are stored as ```spine_dataset``` attributes.

In [ ]:
# TODO: this still points to a private method, need to create a public one
spine_dataset._collect_spines_and_dendrites_data(save=True)

In [ ]:
print(f"Number of spines detected: {len(spine_dataset.spines_data)}")

### Visual inspection of spine locations

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()

spyne.plot.scatter(spine_dataset.spines_data, ax=ax)

ax.set(
    aspect='equal',
    xlabel='X (µm)',
    ylabel='Y (µm)',
)
plt.show()

Now, let's collect the fluorescence variation time series from each spine.  By setting ```save=True``` we create ```.h5``` files for $\frac{\Delta F}{F_{0}}$, zscore and timestamps time series for both CA3 and BLA.

In [ ]:
# TODO: this still points to a private method, need to create a public one
spine_dataset._collect_timeseries(save=True)

Next, we detect calcium events among the collected timeseries, using custom ```calcium_event_classifier```. This generates ```.h5``` files containing probability values of a trace containing an event.

In [ ]:
# TODO: this still points to a private method, need to create a public one
spine_dataset._calcium_events_predictions(save=True)

In [ ]:
import matplotlib.colors as mcolors
import numpy as np

fig, ax = plt.subplots()

# Define colormap and normalization
cmap = plt.get_cmap("coolwarm")
norm = mcolors.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)

# Map maximum calcium event probabilities across 5 sweeps to colors
max_predictions = np.max(spine_dataset.calcium_event_probabilities_BLA, axis=1)
spine_colors = [cmap(norm(prediction)) for prediction in max_predictions]

spyne.plot.scatter(
    spine_dataset.spines_data,
    ax=ax,
    scan_angle=True,
    color=spine_colors,
    edgecolor="black",
    linewidth=0.2,
)
ax.set(
    aspect='equal',
    xlabel='X (scan degrees)',
    ylabel='Y (scan degrees)',
)

# Add colorbar
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label('Max. calcium event probability')

plt.show()

In [ ]:
ephy_dataset = spyne.EphyDataset(dataset)

In [ ]:
ephy_dataset.save_metadata()

In [ ]:
ephy_dataset._collect_timeseries(save=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib qt

fig, ax = plt.subplots()

# Plot all sweeps without for loop
for z in range(ephy_dataset.sweep_y.shape[0]):
    ax.plot(ephy_dataset.sweep_x[z].T, ephy_dataset.sweep_y[z].T, color="lightgray", lw=0.8)
ax.plot(ephy_dataset.sweep_x[0][0], np.mean(ephy_dataset.sweep_y, axis=(0, 1)), color="blue", lw=2)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib qt

fig, ax = plt.subplots()

offset = 0
increment = 100

ax.axvline(x=1.324, color="red", lw=1,)

# Plot all sweeps without for loop
for z in range(ephy_dataset.sweep_y.shape[0]):
    for sweep_n in range(ephy_dataset.sweep_y.shape[1]):
        ax.plot(ephy_dataset.sweep_x[z][sweep_n], ephy_dataset.sweep_y[z][sweep_n] + offset, color="lightgray", lw=0.8)
        offset += increment

ax.set_xlim(0, 1.8)

In [ ]:
# find the indices of the max probability in the 3d array
index = np.unravel_index(np.argmax(spine_dataset.calcium_event_probabilities_BLA, axis=None), spine_dataset.calcium_event_probabilities_BLA.shape)
print(f"Max probability at spine index {index[0]}, sweep index {index[1]} with probability {spine_dataset.calcium_event_probabilities_BLA[index]:.4f}")

In [ ]:
spine_dataset.zscores_BLA
